## Retrieving and preparing the Data


In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

keras.utils.set_random_seed(42)

In [2]:
from google.colab import files
uploaded = files.upload()

import io
train_df = pd.read_csv(io.BytesIO(uploaded['lyric_genre_train.csv']), index_col=0).astype(str)
test_df  = pd.read_csv(io.BytesIO(uploaded['lyric_genre_test.csv']),  index_col=0).astype(str)
val_df   = pd.read_csv(io.BytesIO(uploaded['lyric_genre_val.csv']),   index_col=0).astype(str)

print(f"""
Train samples: {train_df.shape[0]}
Validation samples: {val_df.shape[0]}
Test samples: {test_df.shape[0]}
""")

Saving lyric_genre_test.csv to lyric_genre_test.csv
Saving lyric_genre_train.csv to lyric_genre_train.csv
Saving lyric_genre_val.csv to lyric_genre_val.csv

Train samples: 16883
Validation samples: 16331
Test samples: 17744



In [3]:
train_df.head()

,Lyric,Genre
0,"Oh, girl. I can't get ready (Can't get ready f...",Pop
1,We met on a rainy evening in the summertime. D...,Pop
2,We carried you in our arms. On Independence Da...,Rock
3,I know he loved you. A long time ago. I ain't ...,Pop
4,Paralysis through analysis. Yellow moral uncle...,Rock


In [4]:
train_df.tail()

,Lyric,Genre
16879,"Speed along the highway, honey I want it my wa...",Rock
16880,"New Jersey Demo. House of fire. House of fire,...",Rock
16881,No demands. Just pleasurable sensations. Hand ...,Pop
16882,Written by Billy Burnette and Rafe Van Hoy.. ....,Rock
16883,Huddled in the safety of a pseudo silk kimono....,Rock


In [5]:
train_df['Genre'].value_counts() / train_df.shape[0]

,count
Genre,
Rock,0.547533
Pop,0.300835
Hip Hop,0.151632


In [6]:
y_train = pd.get_dummies(train_df['Genre']).to_numpy()
y_val = pd.get_dummies(val_df['Genre']).to_numpy()
y_test = pd.get_dummies(test_df['Genre']).to_numpy()

In [7]:
y_train

array([[False,  True, False],
       [False,  True, False],
       [False, False,  True],
       ...,
       [False,  True, False],
       [False, False,  True],
       [False, False,  True]])

## Model 1: Baseline Model (Bag of Words)


In [8]:
max_tokens = 5000
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="multi_hot")


In [9]:
text_vectorization.adapt(train_df['Lyric'])

In [10]:
text_vectorization.get_vocabulary()[-20:]

[np.str_('nevermind'),
 np.str_('meth'),
 np.str_('messy'),
 np.str_('melting'),
 np.str_('marijuana'),
 np.str_('lethal'),
 np.str_('knocks'),
 np.str_('ignite'),
 np.str_('hidin'),
 np.str_('helpin'),
 np.str_('hangs'),
 np.str_('gut'),
 np.str_('gotcha'),
 np.str_('fonte'),
 np.str_('flee'),
 np.str_('fantastic'),
 np.str_('engines'),
 np.str_('elevator'),
 np.str_('dressing'),
 np.str_('draws')]

In [11]:
X_train = text_vectorization(train_df['Lyric'])
X_val = text_vectorization(val_df['Lyric'])
X_test = text_vectorization(test_df['Lyric'])

In [12]:
X_train

<tf.Tensor: shape=(16883, 5000), dtype=int64, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 0, ..., 0, 0, 0]])>

1-hidden layer NN with just 8 neurons in the hidden layer.

In [13]:
inputs = keras.Input(shape=(max_tokens, ))
x = keras.layers.Dense(8, activation="relu")(inputs)
outputs = keras.layers.Dense(3, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 5000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │        40,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,035 (156.39 KB)

 Trainable params: 40,035 (156.39 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

In [15]:
model.fit(x=X_train, y=y_train,
          validation_data=(X_val, y_val),
          epochs=10,
          batch_size=32)

Epoch 1/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7069 - loss: 0.6753 - val_accuracy: 0.7409 - val_loss: 0.6011
Epoch 2/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7729 - loss: 0.5314 - val_accuracy: 0.7347 - val_loss: 0.6058
Epoch 3/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8022 - loss: 0.4703 - val_accuracy: 0.7260 - val_loss: 0.6281
Epoch 4/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8231 - loss: 0.4225 - val_accuracy: 0.7175 - val_loss: 0.6588
Epoch 5/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8396 - loss: 0.3825 - val_accuracy: 0.7109 - val_loss: 0.6950
Epoch 6/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8589 - loss: 0.3469 - val_accuracy: 0.7044 - val_loss: 0.7378
Epoch 7/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8716 - loss: 0.3164 - val_accuracy: 0.6981 - val_loss: 0.7865
Epoch 8/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8857 - loss: 0.2881 - val_accuracy: 0.

In [16]:
model.evaluate(x=X_test, y=y_test)

555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6762 - loss: 1.0041


[1.0040628910064697, 0.6762285828590393]

## Model 2: Word Embeddings — Frozen

In [17]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

--2026-05-16 02:33:18--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-05-16 02:33:19--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-05-16 02:33:19--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [18]:
embedding_dim = 100
path_to_glove_file = f"glove.6B.{embedding_dim}d.txt"

embeddings_index = {}
with open(path_to_glove_file) as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors.")

Found 400000 word vectors.


In [19]:
embeddings_index["movie"]

array([ 0.38251  ,  0.14821  ,  0.60601  , -0.51533  ,  0.43992  ,
        0.061053 , -0.62716  , -0.025385 ,  0.1643   , -0.22101  ,
        0.14423  , -0.37213  , -0.21683  , -0.08895  ,  0.097904 ,
        0.6561   ,  0.64455  ,  0.47698  ,  0.83849  ,  1.6486   ,
        0.88922  , -0.1181   , -0.012465 , -0.52082  ,  0.77854  ,
        0.48723  , -0.014991 , -0.14127  , -0.34747  , -0.29595  ,
        0.1028   ,  0.57191  , -0.045594 ,  0.026443 ,  0.53816  ,
        0.32257  ,  0.40788  , -0.043599 , -0.146    , -0.48346  ,
        0.32036  ,  0.55086  , -0.76259  ,  0.43269  ,  0.61753  ,
       -0.36503  , -0.60599  , -0.79615  ,  0.3929   , -0.23668  ,
       -0.34719  , -0.61201  ,  0.54747  ,  0.94812  ,  0.20941  ,
       -2.7771   , -0.6022   ,  0.8495   ,  1.2549   ,  0.017893 ,
       -0.041901 ,  2.1147   , -0.026618 , -0.28104  ,  0.68124  ,
       -0.14165  ,  0.99249  ,  0.49879  , -0.67538  ,  0.6417   ,
        0.42303  , -0.27913  ,  0.063403 ,  0.68909  , -0.3618

In [20]:
embeddings_index["film"]

array([ 0.19916 , -0.049702,  0.24579 , -0.32281 ,  0.89768 , -0.1278  ,
       -0.49506 ,  0.20814 , -0.20046 , -0.20604 ,  0.038292, -0.67277 ,
       -0.12689 , -0.18766 , -0.10277 ,  0.73128 ,  0.82408 ,  0.087288,
        0.69255 ,  1.3107  ,  0.49113 , -0.38097 ,  0.24338 , -0.27813 ,
        0.62506 ,  0.35978 ,  0.42041 , -0.24529 ,  0.14861 , -0.26726 ,
       -0.56262 ,  0.63843 , -0.54153 ,  0.36537 ,  0.20545 , -0.16604 ,
        0.72434 ,  0.29961 , -0.42501 , -0.35932 , -0.089288,  0.48752 ,
       -1.0927  ,  0.88818 ,  0.89941 , -0.7541  , -0.35492 , -0.76396 ,
        0.27468 ,  0.2757  , -0.48152 , -0.41399 ,  0.64489 ,  1.148   ,
       -0.29131 , -2.9387  , -0.83162 ,  0.95586 ,  1.1623  , -0.42502 ,
        0.15486 ,  2.2326  , -0.31339 , -0.030228,  0.79802 , -0.41302 ,
        0.72885 ,  0.7296  , -0.31909 ,  0.8956  ,  0.34625 ,  0.2923  ,
        0.40056 ,  0.78985 , -0.43999 ,  0.24698 , -0.46548 ,  0.055886,
       -0.62603 , -0.036487, -0.65429 ,  0.10563 , 

load the GloVe embeddings into the model then train the model

In [21]:
max_length = 300 #90% of songs
max_tokens = 5000

text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)

In [22]:
text_vectorization.adapt(train_df['Lyric'])

In [23]:
X_train = text_vectorization(train_df['Lyric'])
X_val = text_vectorization(val_df['Lyric'])
X_test = text_vectorization(test_df['Lyric'])

In [24]:
X_train

<tf.Tensor: shape=(16883, 300), dtype=int64, numpy=
array([[  41,   84,    3, ...,   22,  715,    4],
       [  19,  672,   13, ...,    0,    0,    0],
       [  19, 2488,    4, ...,    0,    0,    0],
       ...,
       [  33,    1,   32, ...,    0,    0,    0],
       [1270,  100, 1870, ...,    0,    0,    0],
       [   1,   11,    2, ...,    0,    0,    0]])>

The above creates a lookup table that maps integers to the corresponding word vectors

In [28]:
vocabulary = text_vectorization.get_vocabulary()

counter = 0
embedding_matrix = np.zeros((max_tokens, embedding_dim), dtype="float32")

for token_id, word in enumerate(vocabulary):
    if token_id >= max_tokens:
        break
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[token_id] = embedding_vector
    else:
        counter += 1

print("embedding_matrix shape:", embedding_matrix.shape)
print("Words in vocab (within max_tokens) missing from GloVe:", counter)

embedding_matrix shape: (5000, 100)
Words in vocab (within max_tokens) missing from GloVe: 165


feed the matrix as the initial weights of the Embedding layer.

In [29]:
embedding_layer = keras.layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer= keras.initializers.Constant(embedding_matrix),
    trainable=False,
)

In [32]:
print(f'There are {counter} words on our vocabulary not present in the Glove embedding')
print(f'This is roughly {counter / max_tokens * 100:.2f}% of the vocabulary')
print('These words will be represented by a vector of 0 in all entries in the embedding')

There are 165 words on our vocabulary not present in the Glove embedding
This is roughly 3.30% of the vocabulary
These words will be represented by a vector of 0 in all entries in the embedding


below now builds a Neural Network with an embedding layer after its input layer



In [36]:
inputs = keras.Input(shape=(max_length,))
embedded = embedding_layer(inputs) # 300 x 100 table comes out
embedded = keras.layers.GlobalAveragePooling1D()(embedded) # 100-element vector
x = keras.layers.Dense(8)(embedded)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(3, activation="softmax")(x)

model_two = keras.Model(inputs, outputs)

model_two.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 300, 100)       │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 100)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 8)              │           808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 500,835 (1.91 MB)

 Trainable params: 835 (3.26 KB)

 Non-trainable params: 500,000 (1.91 MB)

In [37]:
model_two.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

In [38]:
# Fit model
model_two.fit(x=X_train, y=y_train,
          validation_data=(X_val, y_val),
          epochs=10,
          batch_size=32,)

Epoch 1/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.5346 - loss: 0.9944 - val_accuracy: 0.5946 - val_loss: 0.9059
Epoch 2/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.5913 - loss: 0.8846 - val_accuracy: 0.6417 - val_loss: 0.8246
Epoch 3/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.6167 - loss: 0.8385 - val_accuracy: 0.6554 - val_loss: 0.7921
Epoch 4/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.6246 - loss: 0.8215 - val_accuracy: 0.6646 - val_loss: 0.7776
Epoch 5/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6320 - loss: 0.8148 - val_accuracy: 0.6702 - val_loss: 0.7699
Epoch 6/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.6374 - loss: 0.8070 - val_accuracy: 0.6719 - val_loss: 0.7646
Epoch 7/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6409 - loss: 0.8023 - val_accuracy: 0.6722 - val_loss: 0.7592
Epoch 8/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6445 - loss: 0.7990 - val_accuracy:

In [39]:
model_two.evaluate(x=X_test, y=y_test)

555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6696 - loss: 0.7637


[0.763694167137146, 0.6696348190307617]

## Model 3: Word Embeddings — Fine-Tuned

I set trainable=True so the embedding vectors in the embedding layer can be fine-tuned during model training rather than staying fixed at their initial values.


In [41]:
embedding_layer = keras.layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=True,
)

inputs = keras.Input(shape=(max_length,))
embedded = embedding_layer(inputs)
embedded = keras.layers.GlobalAveragePooling1D()(embedded)
x = keras.layers.Dense(8)(embedded)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(3, activation="softmax")(x)

model_three = keras.Model(inputs, outputs)
model_three.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 300, 100)       │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_4      │ (None, 100)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 8)              │           808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 500,835 (1.91 MB)

 Trainable params: 500,835 (1.91 MB)

 Non-trainable params: 0 (0.00 B)

In [42]:
model_three.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])


In [43]:
# Fit model
model_three.fit(x=X_train, y=y_train,
          validation_data=(X_val, y_val),
          epochs=10,
          batch_size=32,)

Epoch 1/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.6265 - loss: 0.8459 - val_accuracy: 0.7013 - val_loss: 0.7301
Epoch 2/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.6938 - loss: 0.7375 - val_accuracy: 0.7192 - val_loss: 0.6759
Epoch 3/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.7090 - loss: 0.6867 - val_accuracy: 0.7288 - val_loss: 0.6460
Epoch 4/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - accuracy: 0.7273 - loss: 0.6528 - val_accuracy: 0.7292 - val_loss: 0.6438
Epoch 5/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.7380 - loss: 0.6289 - val_accuracy: 0.7191 - val_loss: 0.6660
Epoch 6/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.7497 - loss: 0.6089 - val_accuracy: 0.7251 - val_loss: 0.6431
Epoch 7/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.7557 - loss: 0.5936 - val_accuracy: 0.7261 - val_loss: 0.6447
Epoch 8/10
528/528 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.7628 - loss: 0.5752 - v

In [44]:
model_three.evaluate(x=X_test, y=y_test)

555/555 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7135 - loss: 0.6855


[0.6855388283729553, 0.7134805917739868]